# 04_subtitle.ipynb — UC-4 자막 다운로드

`subtitle()` 함수와 트랙/포맷 선택, 재사용, 예외 검증.


In [ ]:
import os
import youtube_kit as yk

from pathlib import Path
# 루트의 outputs 에 저장한다. notebooks 에서 열어도, 루트에서 실행해도 같은 곳이다
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
out_dir = str(ROOT / "outputs")
os.makedirs(out_dir, exist_ok=True)


## T-1 (bWPXADZylm0): 수동 자막 다운로드 (기본 srt)


In [2]:
# info() 결과에서 자막 트랙 선택 (하드코딩하지 않음)
info_t1 = yk.info("bWPXADZylm0", out_dir=out_dir)
manual_track = next(s["id"] for s in info_t1["subtitles"] if s["kind"] == "manual" and s["key"] == "ko")
print("Selected manual track:", manual_track)

res_sub = yk.subtitle("bWPXADZylm0", track=manual_track, out_dir=out_dir)
print("subtitle() result:", res_sub)
assert res_sub["status"] in ("ok", "reused")
assert res_sub["path"] == os.path.join(out_dir, "bWPXADZylm0.manual.ko.srt")
assert os.path.exists(res_sub["path"])


Selected manual track: manual:ko
subtitle() result: {'status': 'reused', 'video_id': 'bWPXADZylm0', 'url': 'https://www.youtube.com/watch?v=bWPXADZylm0', 'path': 'outputs\\bWPXADZylm0.manual.ko.srt'}


## 파일 재사용 및 overwrite 확인


In [3]:
res_re = yk.subtitle("bWPXADZylm0", track=manual_track, out_dir=out_dir)
assert res_re["status"] == "reused"

res_ow = yk.subtitle("bWPXADZylm0", track=manual_track, out_dir=out_dir, overwrite=True)
assert res_ow["status"] == "ok"
assert os.path.exists(res_ow["path"])
print("Reuse and overwrite verified.")


Reuse and overwrite verified.


## 자동 자막 및 다른 형식 (json3) 다운로드


In [4]:
auto_track = next(s["id"] for s in info_t1["subtitles"] if s["kind"] == "auto" and s["key"] == "ko-orig")
print("Selected auto track:", auto_track)

res_json3 = yk.subtitle("bWPXADZylm0", track=auto_track, fmt="json3", out_dir=out_dir)
print("json3 subtitle result:", res_json3)
assert res_json3["status"] in ("ok", "reused")
assert res_json3["path"] == os.path.join(out_dir, "bWPXADZylm0.auto.ko-orig.json3")
assert os.path.exists(res_json3["path"])


Selected auto track: auto:ko-orig
json3 subtitle result: {'status': 'reused', 'video_id': 'bWPXADZylm0', 'url': 'https://www.youtube.com/watch?v=bWPXADZylm0', 'path': 'outputs\\bWPXADZylm0.auto.ko-orig.json3'}


## T-2 (xyTPUdJhxLM): 긴 트랙 키 선택 및 파일명 규칙 확인


In [5]:
info_t2 = yk.info("xyTPUdJhxLM", out_dir=out_dir)
t2_track = next(s["id"] for s in info_t2["subtitles"] if "1tAUxhK-Qig" in s["key"])
print("Selected T-2 track:", t2_track)
kind, key = t2_track.split(":", 1)
expected_filename = f"xyTPUdJhxLM.{kind}.{key}.srt"
print("Expected filename:", expected_filename)
assert ":" not in expected_filename


Selected T-2 track: manual:ko-1tAUxhK-Qig
Expected filename: xyTPUdJhxLM.manual.ko-1tAUxhK-Qig.srt


## T-3 (VbD8ITrJ6lg): 수동 자막 없는 영상에 수동 자막 요청 -> NotAvailable


In [6]:
try:
    yk.subtitle("VbD8ITrJ6lg", track="manual:ko", out_dir=out_dir)
    assert False, "Expected NotAvailable"
except yk.NotAvailable as exc:
    print("T-3 rejected non-existent manual subtitle:", type(exc).__name__)


T-3 rejected non-existent manual subtitle: NotAvailable


## T-12 (DDubtRhOEGw): 자막 없는 영상에 자막 요청 -> NotAvailable


In [7]:
try:
    yk.subtitle("DDubtRhOEGw", track="manual:ko", out_dir=out_dir)
    assert False, "Expected NotAvailable"
except yk.NotAvailable as exc:
    print("T-12 rejected subtitle request:", type(exc).__name__)


T-12 rejected subtitle request: NotAvailable


## 트랙이 제공하지 않는 형식 요청 -> NotAvailable


In [8]:
try:
    yk.subtitle("bWPXADZylm0", track=manual_track, fmt="unsupported_fmt", out_dir=out_dir)
    assert False, "Expected NotAvailable"
except yk.NotAvailable as exc:
    print("Unsupported format rejected:", type(exc).__name__)


Unsupported format rejected: NotAvailable


## 잘못된 트랙 ID 형식 -> InvalidInput


In [9]:
try:
    yk.subtitle("bWPXADZylm0", track="ko", out_dir=out_dir)
    assert False, "Expected InvalidInput"
except yk.InvalidInput as exc:
    print("Invalid track format rejected:", type(exc).__name__)

try:
    yk.subtitle("https://www.youtube.com/playlist?list=PLyPc3E0Xm540Yfdln9qlVyyrqRME80jJm", track="manual:ko", out_dir=out_dir)
    assert False, "Expected InvalidInput for playlist URL"
except yk.InvalidInput as exc:
    print("Playlist URL rejected in subtitle():", type(exc).__name__)


Invalid track format rejected: InvalidInput
Playlist URL rejected in subtitle(): InvalidInput


## T-2 · T-7 · T-8 자막 받기

긴 트랙 키를 실제로 받아 파일명을 확인하고, 수동 트랙이 여럿인 영상에서 골라 받는다.

In [ ]:
r = yk.subtitle("xyTPUdJhxLM", track=t2_track, out_dir=out_dir)
assert r["path"] == os.path.join(out_dir, expected_filename) and os.path.exists(r["path"])
print("T-2 downloaded:", r["path"])

for vid, track in [("lbzSN3fFV_Q", "manual:ja"), ("lbzSN3fFV_Q", "manual:pt-BR"), ("KM4Xe6Dlp0Y", "manual:en")]:
    meta = yk.info(vid, out_dir=out_dir)
    assert any(s["id"] == track for s in meta["subtitles"]), track
    r = yk.subtitle(vid, track=track, out_dir=out_dir)
    kind, key = track.split(":", 1)
    assert r["path"] == os.path.join(out_dir, f"{vid}.{kind}.{key}.srt") and os.path.exists(r["path"])
    print(vid, track, r["status"])